# 06 — Frozen full test experiment

This is the first post-freeze allocator experiment.

**No tuning is performed here.** The notebook:
- verifies hashes recorded by 05f;
- loads the frozen payload grid and frozen alpha;
- uses the SRM-teacher-derived local risk model fixed before this test run;
- evaluates `raster`, `random`, `predictability`, `detectability`, and `joint`;
- uses exactly the configured test split;
- checkpoints every case to JSONL and is safe to resume;
- checks exact image/message recovery for every feasible case;
- records a sampled PNG save/reload roundtrip;
- reports both strategy-specific feasibility and common-feasible summaries.

Important wording for the paper: the final split is held out from **allocator / detector selection**. If an earlier engineering baseline notebook accessed the same test images, do not call the split literally “never seen”; document that distinction.


In [ ]:
from pathlib import Path
from time import perf_counter
import json, joblib, yaml
import numpy as np
import pandas as pd

from rdhlab.io import read_gray
from rdhlab.pipeline import load_payload_freeze, prepare_image_context, run_frozen_image_precomputed
from rdhlab.freeze_protocol import (
    sha256_file, stable_id_hash, append_jsonl, load_jsonl, case_key, atomic_write_json
)

config=yaml.safe_load(Path('/workspace/config/experiment.yaml').read_text())
seed=int(config['project']['seed'])
bs=int(config['dataset']['block_size'])
manifest_path=Path(config['dataset']['prepared_manifest'])
manifest=pd.read_csv(manifest_path)
test=manifest[manifest.split=='test'].reset_index(drop=True)

expected_test=int(config['dataset']['expected_splits']['test'])
assert len(test)==expected_test, (len(test),expected_test)
assert test.source_id.astype(str).is_unique, 'Test source_id values must be unique.'

payload_path=Path('/workspace/config/frozen_payloads.json')
allocator_path=Path('/workspace/config/frozen_allocator.json')
risk_path=Path('/workspace/results/models/srm_teacher_local_risk.joblib')
decision_path=Path('/workspace/results/cnn_repair_transfer/decision.json')
rule_path=Path('/workspace/results/cnn_transfer/candidate_alpha_rule.json')
cnn_path=Path('/workspace/results/models/enhanced_residual_cnn_05e.pt')

frozen_payload=load_payload_freeze(payload_path)
payloads=list(map(float,frozen_payload['levels']))
allocator=json.loads(allocator_path.read_text())
assert allocator['status']=='FROZEN_AFTER_05E_TRANSFER_CONFIRMATION'
alpha=float(allocator['alpha'])
assert payloads==list(map(float,allocator['payload_levels']))

# Fail closed if any selection-critical artifact changed after 05f.
prov=allocator['provenance_sha256']
checks={
    '05e_decision_json':decision_path,
    '05d_candidate_alpha_rule_json':rule_path,
    'frozen_payloads_json':payload_path,
    'dataset_manifest_csv':manifest_path,
    'srm_teacher_local_risk_joblib':risk_path,
    'enhanced_residual_cnn_05e_pt':cnn_path,
}
for name,path in checks.items():
    expected=prov.get(name)
    actual=sha256_file(path)
    if expected is not None and actual!=expected:
        raise RuntimeError(f'Frozen provenance mismatch for {name}: expected {expected}, got {actual}')

local_risk=joblib.load(risk_path)
strategies=list(config['allocator']['strategies'])
assert strategies==['raster','random','predictability','detectability','joint'], strategies

out=Path('/workspace/results/frozen_test_final'); out.mkdir(parents=True,exist_ok=True)
context_tag=f"srmteacher_{sha256_file(risk_path)[:12]}_a{alpha:.6f}".replace('.','p')
context_dir=out/'contexts'/context_tag; context_dir.mkdir(parents=True,exist_ok=True)
jsonl_path=out/'per_case.jsonl'
tmp=out/'_roundtrip_check.png'

protocol={
    'status':'POST_FREEZE_TEST',
    'alpha':alpha,
    'payloads':payloads,
    'strategies':strategies,
    'test_images':len(test),
    'expected_cases':len(test)*len(payloads)*len(strategies),
    'test_source_ids_sha256':stable_id_hash(test.source_id.astype(str).tolist()),
    'allocator_sha256':sha256_file(allocator_path),
    'payload_freeze_sha256':sha256_file(payload_path),
    'risk_model_sha256':sha256_file(risk_path),
    'dataset_manifest_sha256':sha256_file(manifest_path),
    'context_tag':context_tag,
    'resume_checkpoint':'per_case.jsonl',
    'selection_note':'No method tuning is permitted from these test results.',
}
protocol_path=out/'test_protocol.json'
if protocol_path.exists():
    old=json.loads(protocol_path.read_text())
    for k in ['alpha','payloads','strategies','test_images','test_source_ids_sha256','allocator_sha256','payload_freeze_sha256','risk_model_sha256','dataset_manifest_sha256','context_tag']:
        if old.get(k)!=protocol.get(k):
            raise RuntimeError(f'Existing test protocol conflicts at {k}: {old.get(k)!r} != {protocol.get(k)!r}')
else:
    atomic_write_json(protocol_path,protocol)

records=load_jsonl(jsonl_path)
keys=[case_key(r['source_id'],r['strategy'],r['target_net_bpp']) for r in records]
if len(keys)!=len(set(keys)):
    raise RuntimeError('Duplicate case keys found in JSONL checkpoint; stop before continuing.')
done=set(keys)
expected_cases=protocol['expected_cases']
print('FROZEN alpha:',alpha)
print('payloads:',payloads)
print('strategies:',strategies)
print('test images:',len(test))
print('completed cases:',len(done),'/',expected_cases)
print('context cache:',context_dir)


In [ ]:
start=perf_counter()
start_done=len(done)
io_every=int(config['frozen_test']['file_io_check_every'])

for i,row in test.iterrows():
    sid=str(row.source_id)
    x=read_gray(row.path)

    ctx_path=context_dir/f'{sid}.joblib'
    if ctx_path.exists():
        ctx=joblib.load(ctx_path)
        if ctx.get('context_tag')!=context_tag or not np.isclose(float(ctx.get('alpha')),alpha):
            raise RuntimeError(f'Stale context cache detected for {sid}')
        orders,block_rows,plans=ctx['orders'],ctx['block_rows'],ctx['plans']
    else:
        orders,block_rows,plans=prepare_image_context(x,sid,local_risk,alpha,bs,seed)
        joblib.dump({
            'context_tag':context_tag,'alpha':alpha,
            'orders':orders,'block_rows':block_rows,'plans':plans
        },ctx_path)

    for bpp in payloads:
        for strategy in strategies:
            key=case_key(sid,strategy,bpp)
            if key in done:
                continue

            io_check=(i % io_every == 0)
            rr=run_frozen_image_precomputed(
                x,sid,bpp,strategy,orders,block_rows,bs,seed,
                io_check,tmp if io_check else None,plans=plans
            )
            rr['image_index']=int(i)
            rr['file_io_checked']=bool(io_check)
            rr['allocator_alpha']=alpha
            rr['context_tag']=context_tag

            if rr['feasible']:
                if not (rr['exact_image'] and rr['exact_message'] and float(rr['ber'])==0.0):
                    raise RuntimeError(f'Reversibility failure: {sid} {strategy} {bpp}')
                if int(rr['net_payload_bits']) != int(rr['target_net_bits']):
                    raise RuntimeError(f'Net payload mismatch: {sid} {strategy} {bpp}')

            append_jsonl(jsonl_path,rr)
            done.add(key)

    if (i+1)%10==0 or (i+1)==len(test):
        elapsed=perf_counter()-start
        newly=len(done)-start_done
        remaining=expected_cases-len(done)
        if newly>0:
            sec_per_case=elapsed/newly
            eta_min=remaining*sec_per_case/60.0
            print(f"{i+1:4d}/{len(test)} images | cases {len(done)}/{expected_cases} | elapsed {elapsed/60:.1f} min | ETA ~{eta_min:.1f} min")
        else:
            print(f"{i+1:4d}/{len(test)} images | all encountered cases already checkpointed")

if tmp.exists():
    tmp.unlink()
print('Run loop complete. Checkpoint:',jsonl_path)


In [ ]:
records=load_jsonl(jsonl_path)
df=pd.DataFrame(records)
expected=len(test)*len(payloads)*len(strategies)
assert len(df)==expected, (len(df),expected)

df['source_id']=df.source_id.astype(str)
df['target_net_bpp']=df.target_net_bpp.astype(float)
df['feasible']=df.feasible.astype(bool)

keys=df.apply(lambda r: case_key(r.source_id,r.strategy,r.target_net_bpp),axis=1)
assert keys.is_unique
assert set(df.source_id)==set(test.source_id.astype(str))
assert set(df.strategy)==set(strategies)
assert set(np.round(df.target_net_bpp,9))==set(np.round(payloads,9))

ok=df[df.feasible].copy()
assert ok.exact_image.fillna(False).astype(bool).all()
assert ok.exact_message.fillna(False).astype(bool).all()
assert (ok.ber.fillna(1).astype(float)==0.0).all()
assert (ok.net_payload_bits.astype(int)==ok.target_net_bits.astype(int)).all()

df.to_csv(out/'per_image.csv',index=False)

# Strategy-specific feasibility plus metrics on each strategy's feasible cases.
feas=(df.groupby(['strategy','target_net_bpp'],as_index=False)
      .agg(n=('source_id','size'),feasible_n=('feasible','sum'),feasible_fraction=('feasible','mean')))
metric_cols=['actual_net_bpp','psnr','ssim','encode_ms','decode_ms','used_blocks','changed_pixels',
             'mean_abs_change','selected_predictability_mean','selected_detectability_risk_mean']
mets=(ok.groupby(['strategy','target_net_bpp'],as_index=False)[metric_cols]
      .agg(['mean','median']))
mets.columns=['_'.join([x for x in c if x]) if isinstance(c,tuple) else c for c in mets.columns]
# groupby(as_index=False)+multi-agg can vary by pandas; rebuild keys if needed
if 'strategy_' in mets.columns:
    mets=mets.rename(columns={'strategy_':'strategy','target_net_bpp_':'target_net_bpp'})
summary_all=feas.merge(mets,on=['strategy','target_net_bpp'],how='left')
summary_all.to_csv(out/'summary_all.csv',index=False)

# Common-feasible set per payload: same covers for every strategy comparison.
common_rows=[]
for bpp in payloads:
    sub=df[np.isclose(df.target_net_bpp,bpp)]
    pivot=sub.pivot(index='source_id',columns='strategy',values='feasible')
    pivot=pivot.reindex(columns=strategies)
    ids=pivot.index[pivot.fillna(False).all(axis=1)].astype(str).tolist()
    common_rows.extend({'source_id':sid,'target_net_bpp':float(bpp)} for sid in ids)
common=pd.DataFrame(common_rows)
common.to_csv(out/'common_feasible_ids.csv',index=False)

common_key=set((str(r.source_id),float(r.target_net_bpp)) for _,r in common.iterrows())
mask=df.apply(lambda r:(str(r.source_id),float(r.target_net_bpp)) in common_key,axis=1)
cdf=df[mask & df.feasible].copy()

summary_common=(cdf.groupby(['strategy','target_net_bpp'],as_index=False)
    .agg(
        n=('source_id','size'),
        actual_net_bpp_mean=('actual_net_bpp','mean'),
        psnr_mean=('psnr','mean'),psnr_median=('psnr','median'),
        ssim_mean=('ssim','mean'),ssim_median=('ssim','median'),
        encode_ms_median=('encode_ms','median'),decode_ms_median=('decode_ms','median'),
        used_blocks_mean=('used_blocks','mean'),
        changed_pixels_mean=('changed_pixels','mean'),
        selected_P_mean=('selected_predictability_mean','mean'),
        selected_D_mean=('selected_detectability_risk_mean','mean'),
    ))
summary_common.to_csv(out/'summary_common_feasible.csv',index=False)

common_counts=(common.groupby('target_net_bpp').size().to_dict() if len(common) else {})
complete={
    'status':'COMPLETE',
    'cases':int(len(df)),
    'expected_cases':int(expected),
    'feasible_cases':int(df.feasible.sum()),
    'exact_recovery_feasible_cases':int(len(ok)),
    'file_io_checked_cases':int(df.file_io_checked.fillna(False).astype(bool).sum()),
    'common_feasible_images_by_payload':{str(k):int(v) for k,v in common_counts.items()},
    'alpha':alpha,
    'payloads':payloads,
    'strategies':strategies,
    'allocator_sha256':sha256_file(allocator_path),
    'risk_model_sha256':sha256_file(risk_path),
    'test_source_ids_sha256':stable_id_hash(test.source_id.astype(str).tolist()),
    'no_retuning_permitted':True,
}
atomic_write_json(out/'test_run_complete.json',complete)

print(json.dumps(complete,indent=2))
display(summary_all)
display(summary_common)


## Stop point after 06

When `test_run_complete.json` says `COMPLETE`, the RDH/reversibility/distortion stage is finished.

Do **not** change the frozen allocator based on these results.

The next stage should be a revised `07` that evaluates detectability on the same common-feasible test subsets using independent detectors. The enhanced residual CNN from 05e can be reused as a pre-specified independent detector; any adaptive/retrained detector must be clearly separated as an additional robustness analysis.
